# SFT / Information Tuning in Colab (End-to-End)
This notebook teaches you supervised fine-tuning (SFT) using LoRA (and QLoRA 4-bit if you want).

You will learn:
1) How instruction tuning differs from "traditional tuning"
2) How to build prompt+response training text
3) The key trick: label-masking so the model is trained to predict ONLY the answer
4) How to load a base model, attach LoRA adapters, train, evaluate, and run inference

You will produce:
- A trained LoRA adapter folder (small)
- A tokenizer folder
- A simple inference demo


In [ ]:
import torch, os, platform, subprocess, textwrap, math, random
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
else:
    print("No GPU found. In Colab: Runtime -> Change runtime type -> GPU")


Python: 3.12.12
Torch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
Capability: (7, 5)


In [ ]:
# Colab tip: If you re-run installs, restart runtime to avoid weird import states.
# Runtime -> Restart runtime

!pip -q install -U transformers datasets accelerate peft bitsandbytes sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Make runs repeatable-ish (not perfect, but helps).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Less noisy logs
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
## Choose a Mode

MODE = "SFT_INSTRUCTION"
#- Uses Alpaca instruction dataset: instruction (+ optional input) -> output

MODE = "INFO_TUNE"
#- Uses a tiny custom dataset (facts -> Q/A)
#- This is useful when you want the model to memorize *your* domain facts/policies.


In [ ]:
# ====== MODE SWITCH ======
MODE = "SFT_INSTRUCTION"   # or "INFO_TUNE"

# ====== MODEL CHOICE ======
# Small model so Colab GPU doesn't cry. TinyLlama is a popular small Llama-style model.
# Model page exists on Hugging Face :contentReference[oaicite:2]{index=2}
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ====== TRAINING SIZE / SPEED SETTINGS ======
MAX_SEQ_LEN = 512              # increase to 1024 if your GPU can handle it
TRAIN_SAMPLES = 6000           # keep modest for Colab
EVAL_SAMPLES  = 500

# Batch sizing: actual batch = per_device_batch * grad_accum
PER_DEVICE_TRAIN_BATCH = 2
GRAD_ACCUM_STEPS = 8

# Keep it short for demo; you can raise later.
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

OUTPUT_DIR = "/content/sft_lora_runs"

print("MODE:", MODE)
print("MODEL:", MODEL_ID)
print("MAX_SEQ_LEN:", MAX_SEQ_LEN)
print("OUTPUT_DIR:", OUTPUT_DIR)


MODE: SFT_INSTRUCTION
MODEL: TinyLlama/TinyLlama-1.1B-Chat-v1.0
MAX_SEQ_LEN: 512
OUTPUT_DIR: /content/sft_lora_runs


## What SFT does (plain language)
We show the model a prompt and the correct answer.
We train it to predict the answer tokens.

Key trick: **label masking**
- We do NOT want the model to “learn” to predict the instruction text.
- We only want it to learn the response.
So we set labels for prompt tokens to -100 (ignored by loss), and labels for answer tokens to real token IDs.

## Tiny math view (cross entropy)
Loss sums over time-steps:
L = - Σ log p(y_t | context)

But with masking:
- prompt positions contribute 0 to the loss (ignored)
- answer positions contribute normally

## Real-world analogy
It’s like giving a student:
- a question sheet (instruction)
- and grading ONLY the final answers, not whether they can rewrite the question.

## One-line summary
SFT = "predict the response", with prompt tokens masked out from training loss.


In [ ]:
def load_sft_dataset():
    # Alpaca dataset exists on HF
    ds = load_dataset("tatsu-lab/alpaca", split="train")
    # Keep just a subset for Colab
    ds = ds.shuffle(seed=SEED)
    ds_train = ds.select(range(min(TRAIN_SAMPLES, len(ds))))
    ds_eval  = ds.select(range(min(EVAL_SAMPLES,  len(ds))))

    return ds_train, ds_eval

def build_info_tune_dataset():
    # This is a tiny "knowledge pack". Replace with your real facts.
    # The idea: make the model answer questions using these facts.
    # If you want it to be robust, write 5-20 question variations per fact.
    records = [
        {
            "instruction": "Answer using the given policy fact.",
            "input": "FACT: Our lab meeting is every Wednesday at 5 PM IST.\nQUESTION: When is our lab meeting?",
            "output": "Your lab meeting is every Wednesday at 5 PM IST."
        },
        {
            "instruction": "Answer using the given policy fact.",
            "input": "FACT: Reimbursement limit for travel is 10,000 INR per trip.\nQUESTION: What is the travel reimbursement limit?",
            "output": "The travel reimbursement limit is 10,000 INR per trip."
        },
        {
            "instruction": "Answer using the given fact.",
            "input": "FACT: The internal code name for Project SPD is 'PostHocRefine'.\nQUESTION: What is the code name for Project SPD?",
            "output": "The internal code name for Project SPD is PostHocRefine."
        },
    ]

    ds = Dataset.from_list(records).shuffle(seed=SEED)
    # In tiny datasets, eval is tiny too (just for sanity).
    n = len(ds)
    n_train = max(1, int(0.8 * n))
    ds_train = ds.select(range(n_train))
    ds_eval  = ds.select(range(n_train, n))
    return ds_train, ds_eval

if MODE == "SFT_INSTRUCTION":
    train_raw, eval_raw = load_sft_dataset()
else:
    train_raw, eval_raw = build_info_tune_dataset()

print("Train rows:", len(train_raw))
print("Eval rows:", len(eval_raw))
print(train_raw[0])


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Train rows: 6000
Eval rows: 500
{'instruction': 'What would be the best type of exercise for a person who has arthritis?', 'input': '', 'output': 'For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat would be the best type of exercise for a person who has arthritis?\n\n### Response:\nFor someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.'}


In [ ]:
def format_prompt(example):
    """
    Alpaca-style fields:
    - instruction: the task
    - input: optional extra context
    - output: the ground-truth answer

    We convert them into one training text:
      [PROMPT PART][ANSWER PART]

    We'll later mask labels so only ANSWER contributes to loss.
    """
    instruction = (example.get("instruction") or "").strip()
    inp         = (example.get("input") or "").strip()
    output      = (example.get("output") or "").strip()

    # A simple, readable format.
    # You can change this, but keep it consistent across train/eval/inference.
    if inp:
        prompt = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{inp}\n\n"
            "### Response:\n"
        )
    else:
        prompt = (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
        )

    return prompt, output


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# Many Llama-style tokenizers don't have a pad token by default.
# Trainer needs padding for batching.
if tokenizer.pad_token is None:
    # Common hack: reuse eos token as pad token (works fine for training batches).
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer pad_token:", tokenizer.pad_token, "id:", tokenizer.pad_token_id)
print("Tokenizer eos_token:", tokenizer.eos_token, "id:", tokenizer.eos_token_id)
print("Tokenizer bos_token:", tokenizer.bos_token, "id:", tokenizer.bos_token_id)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer pad_token: </s> id: 2
Tokenizer eos_token: </s> id: 2
Tokenizer bos_token: <s> id: 1


In [ ]:
def tokenize_with_mask(example):
    """
    We build:
    - full_text  = (prompt + answer + eos)
    - prompt_text = prompt only

    Then:
    - input_ids = tokenize(full_text)
    - labels = input_ids copy
    - labels[0:len(prompt_tokens)] = -100  (ignore prompt in loss)

    This makes training focus on predicting the answer.
    """

    prompt, answer = format_prompt(example)

    # Always end answers with EOS so the model learns "stop here".
    # If eos_token is None, we'll just not add it.
    eos = tokenizer.eos_token if tokenizer.eos_token else ""
    full_text = prompt + answer + eos

    # Tokenize prompt alone to find where answer begins
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )["input_ids"]

    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]

    labels = input_ids.copy()

    # Mask prompt positions
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


In [ ]:
train_tok = train_raw.map(tokenize_with_mask, remove_columns=train_raw.column_names)
eval_tok  = eval_raw.map(tokenize_with_mask,  remove_columns=eval_raw.column_names)

print(train_tok[0].keys())

# --- Sanity check: show where masking happens ---
def debug_one(i=0):
    row = train_tok[i]
    ids = row["input_ids"]
    labs = row["labels"]

    # Find first unmasked label index (start of answer)
    first_answer_pos = next((j for j, x in enumerate(labs) if x != -100), None)

    print("Total tokens:", len(ids))
    print("First answer token index:", first_answer_pos)

    # Print a small window of tokens around boundary
    left = max(0, (first_answer_pos or 0) - 20)
    right = min(len(ids), (first_answer_pos or 0) + 40)

    print("\n--- TOKENS AROUND THE BOUNDARY ---")
    for j in range(left, right):
        tok = tokenizer.decode([ids[j]])
        mark = "ANSWER" if labs[j] != -100 else "prompt"
        print(f"{j:04d} | {mark:6s} | {repr(tok)}")

debug_one(0)


Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])
Total tokens: 84
First answer token index: 30

--- TOKENS AROUND THE BOUNDARY ---
0010 | prompt | 'type'
0011 | prompt | 'of'
0012 | prompt | 'exercise'
0013 | prompt | 'for'
0014 | prompt | 'a'
0015 | prompt | 'person'
0016 | prompt | 'who'
0017 | prompt | 'has'
0018 | prompt | 'ar'
0019 | prompt | 'th'
0020 | prompt | 'rit'
0021 | prompt | 'is'
0022 | prompt | '?'
0023 | prompt | '\n'
0024 | prompt | '\n'
0025 | prompt | '##'
0026 | prompt | '#'
0027 | prompt | 'Response'
0028 | prompt | ':'
0029 | prompt | '\n'
0030 | ANSWER | 'For'
0031 | ANSWER | 'someone'
0032 | ANSWER | 'with'
0033 | ANSWER | 'ar'
0034 | ANSWER | 'th'
0035 | ANSWER | 'rit'
0036 | ANSWER | 'is'
0037 | ANSWER | ','
0038 | ANSWER | 'the'
0039 | ANSWER | 'best'
0040 | ANSWER | 'type'
0041 | ANSWER | 'of'
0042 | ANSWER | 'exercise'
0043 | ANSWER | 'would'
0044 | ANSWER | 'be'
0045 | ANSWER | 'low'
0046 | ANSWER | '-'
0047 | ANSWER | 'imp'
0048 | ANSWER | 'act'
0049

In [ ]:
def collate_fn(batch):
    """
    Batch is a list of dicts with variable-length sequences.
    We pad them to the max length in the batch.
    """
    max_len = max(len(x["input_ids"]) for x in batch)

    def pad_list(lst, pad_value):
        return lst + [pad_value] * (max_len - len(lst))

    input_ids = []
    attention_mask = []
    labels = []

    for x in batch:
        input_ids.append(pad_list(x["input_ids"], tokenizer.pad_token_id))
        attention_mask.append(pad_list(x["attention_mask"], 0))
        labels.append(pad_list(x["labels"], -100))

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


In [ ]:
# Compute dtype choice:
# - bf16 is great if supported
# - otherwise fp16
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",          # common QLoRA setting
    bnb_4bit_use_double_quant=True,     # common QLoRA setting
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# Helps k-bit finetuning (fixes layer norms, etc.)
model = prepare_model_for_kbit_training(model)

# LoRA config: for Llama-like architectures, these module names are typical.
# If you switch to a different model family, target_modules may need changes.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

# Print trainable parameter count (so you see LoRA is small)
def print_trainable_params(m):
    trainable = 0
    total = 0
    for _, p in m.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print(f"Trainable params: {trainable:,}")
    print(f"Total params:     {total:,}")
    print(f"Trainable %:      {100 * trainable / total:.4f}%")

print_trainable_params(model)


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Trainable params: 4,505,600
Total params:     620,111,872
Trainable %:      0.7266%


Sanity Check for one sample

In [ ]:
model.train()

batch = collate_fn([train_tok[0], train_tok[1]])
batch = {k: v.to(model.device) for k, v in batch.items()}

out = model(**batch)
print("Forward ok. Loss:", float(out.loss))


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Forward ok. Loss: 0.9064531326293945


/tmp/ipython-input-2562539483.py:7: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Forward ok. Loss:", float(out.loss))


In [ ]:
# A practical Colab-friendly setup.
# You can raise logging_steps / eval_steps as you like.

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Mixed precision
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),

    logging_steps=20,
    # evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    report_to="none",
    remove_unused_columns=False,  # important because we already provide exact tensors
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collate_fn,
)


In [ ]:
train_result = trainer.train()
print(train_result)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.283099
40,1.305853
60,1.287218
80,1.283182
100,1.291397
120,1.317692
140,1.238680
160,1.309594
180,1.307223
200,1.269807


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=375, training_loss=1.2821539866129557, metrics={'train_runtime': 1088.6636, 'train_samples_per_second': 5.511, 'train_steps_per_second': 0.344, 'total_flos': 4891804726542336.0, 'train_loss': 1.2821539866129557, 'epoch': 1.0})


In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

if "eval_loss" in eval_metrics:
    ppl = math.exp(eval_metrics["eval_loss"]) if eval_metrics["eval_loss"] < 50 else float("inf")
    print("Perplexity (rough):", ppl)


{'eval_loss': 1.1495929956436157, 'eval_runtime': 26.9675, 'eval_samples_per_second': 18.541, 'eval_steps_per_second': 9.27, 'epoch': 1.0}
Perplexity (rough): 3.1569077729630854


In [ ]:
# Save only the adapter (small) — this is the usual LoRA workflow.
adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
tokenizer_path = os.path.join(OUTPUT_DIR, "tokenizer")

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(tokenizer_path)

print("Saved adapter to:", adapter_path)
print("Saved tokenizer to:", tokenizer_path)


Saved adapter to: /content/sft_lora_runs/lora_adapter
Saved tokenizer to: /content/sft_lora_runs/tokenizer


In [ ]:
from peft import PeftModel

# Reload fresh (good habit for inference sanity)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base = PeftModel.from_pretrained(base, adapter_path)
base.eval()

def generate_answer(instruction, inp=""):
    ex = {"instruction": instruction, "input": inp, "output": ""}  # dummy output
    prompt, _ = format_prompt(ex)

    inputs = tokenizer(prompt, return_tensors="pt").to(base.device)

    with torch.no_grad():
        out = base.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text

print(generate_answer("Explain gradient accumulation like I'm new to training LLMs."))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Instruction:
Explain gradient accumulation like I'm new to training LLMs.

### Response:
Gradient accumulation is a technique used in Neural Network Training, which involves using a process called stochastic gradient descent (SGD) to update the weights of a neural network. In SGD, an input signal is passed through the network and its gradient is computed. The gradient is then multiplied by a learning rate and added to the network's weight parameters. This process is repeated until the desired output is obtained. Gradient accumulation helps to reduce the variance in the gradients, allowing for more accurate and stable updates to be made.


## If you are doing IFT seriously
A tiny dataset (3 facts) will NOT reliably “inject knowledge”.

To make information tuning work better:
1) Write 10–50 variations per fact:
   - direct question, indirect question, paraphrase, negative example
2) Mix in “don’t know” examples for out-of-scope questions
3) Keep answers short and consistent
4) Evaluate with:
   - exact-match style checks for factual questions
   - adversarial paraphrases

If you want, replace build_info_tune_dataset() with your real domain pack.
